# Real information, real memory: a working agent

Part 5 left us with a sobering lesson: a small, local model (`mistral:latest` via Ollama) isn't reliable enough to properly decide when to call a tool and faithfully use what it returns — not even when the underlying data is clean and correct. In this part we drop Ollama entirely and use one of the bigger, hosted models on the University of Rennes server (Part 1) instead, and build something that actually works: an agent that looks up real, current information about IMT Atlantique's TAF (Thématiques d'Approfondissement) and remembers what a student tells it, across separate calls.

Two problems to solve, on top of the `research_agent` pattern from Part 5:

1. **Real information**: the model has never seen this year's TAF list — we'll give it a tool that fetches it from IMT Atlantique's own Moodle.
2. **Real memory**: Part 2 already showed an agent forgets everything between calls (the "amnesia" problem). We'll fix this properly this time, with an actual database instead of resending the whole conversation history.

## A tool that reads the real web

IMT Atlantique publishes the current TAF list on a public Moodle page — no login needed, no API key, just an ordinary web page we can fetch with `requests`, exactly like Part 5's Wikipedia calls. This time the content is HTML, not JSON, so we'll pick out the relevant part with [`BeautifulSoup`](https://www.crummy.com/software/BeautifulSoup/) instead of a simple regex.

In [ ]:
# Program 1: a tool that fetches the real, current TAF list

from IPython.display import Markdown, display

import requests
from bs4 import BeautifulSoup
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

HEADERS = {"User-Agent": "Mozilla/5.0 (PLIDOagent-course/1.0; educational use)"}
TAF_PAGE_URL = "https://moodle.imt-atlantique.fr/course/view.php?id=897&section=6"

def fetch_taf_list():
    """Fetch the current list of TAF names in 'Informatique et Réseaux', straight off Moodle."""
    response = requests.get(TAF_PAGE_URL, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")

    # This section of the page is a <li id="section-6"> containing one link per TAF.
    section = soup.find("li", {"id": "section-6"})
    names = []
    for link in section.find_all("a"):
        text = link.get_text(strip=True)
        if text.upper().startswith("TAF") and text not in names:
            names.append(text)
    return names

@function_tool
def imt_taf_list():
    """Return the current list of TAF (Thematique d'Approfondissement) programs in the
    Informatique et Reseaux domain at IMT Atlantique, fetched live from the school's Moodle."""
    return "\n".join(fetch_taf_list())

Now let's give this tool to an agent — but on a model that can actually be trusted to use it. Part 1 already set up a client for the University of Rennes server; we reuse it here.

In [ ]:
# Program 2: a Rennes-hosted agent, using the tool reliably

import os
from dotenv import load_dotenv

load_dotenv(override=True)

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=os.environ["RENNES_API_KEY"])
rennes_model = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

taf_agent = Agent(
    name="TAF Agent",
    instructions="Answer the user's question. If you are not fully confident from memory alone, "
                 "use the imt_taf_list tool before answering.",
    model=rennes_model,
    tools=[imt_taf_list],
)

question = "What TAF programs in Informatique et Reseaux are offered at IMT Atlantique this year?"
result = await Runner.run(taf_agent, question)
display(Markdown(result.final_output))

Run this a few times: unlike Part 5's Program 7 on Ollama, this reliably calls the tool and lists the real, current TAF names — no invented "Thématiques", no skipped tool calls. A bigger, hosted model really does make a difference here, exactly as we saw when testing Program 4's reasoning task against Rennes models in Part 5.

## Memory, or the lack of it

Part 2 already ran into this: an agent has no memory between separate calls unless we do something about it. Let's see it happen again, directly, with this same capable Rennes model — memory isn't a *capability* problem, it's an *architectural* one.

In [ ]:
# Program 3: two separate turns, no memory in between

turn_1 = await Runner.run(taf_agent, "Hi, I'm Marie, and I'm interested in cybersecurity.")
display(Markdown(turn_1.final_output))

turn_2 = await Runner.run(taf_agent, "What TAF would you recommend for me?")
display(Markdown(turn_2.final_output))

Even though `turn_2` uses the very same `taf_agent`, `Runner.run` starts from a blank slate every time — there's no thread connecting the two calls, so the second one has no idea a student named Marie, interested in cybersecurity, ever said anything. Resending the whole conversation history (Part 2's workaround) would technically fix this, but it grows without bound and the model still has to re-read everything, every single time. Let's do this properly instead: an actual database.

## MCP: a standard way to package tools

So far, every tool in this course has been a Python function we wrote ourselves and wrapped with `@function_tool`. That works, but it means reinventing a tool from scratch every time we want a new capability — including something as generic as "remember facts across calls".

**MCP** (Model Context Protocol, introduced by Anthropic) is a standard way to package a tool so any agent framework can use it without custom glue code. An MCP *server* is a small, separate program (often installed and run on the fly with `npx`, Node's package runner) that exposes a set of tools over a simple protocol; our agent code just launches it and gets its tools for free, the same way `@function_tool` exposes one of our own functions. Instead of writing our own "remember a fact" / "recall a fact" tool by hand, we'll launch a ready-made MCP server that does exactly that, backed by a small file-based database.

Running this needs [Node.js](https://nodejs.org/) 18+ installed on your machine (for `npx`); the first run downloads the server package, so it needs internet access once.

In [ ]:
# Program 4: giving the agent a real memory (MCP server, no TAF tool yet -- one thing at a time)

import os
from contextlib import AsyncExitStack

from agents.mcp import MCPServerStdio

memory_path = os.path.abspath(os.path.join(os.getcwd(), "memory", "part6_relations.json"))
os.makedirs(os.path.dirname(memory_path), exist_ok=True)

memory_instructions = (
    "Before answering anything, always call the memory tool read_graph (no arguments) to load "
    "what you currently remember about this student. If the student's message contains a new "
    "fact about themselves (their name, or an interest), store it right away with the memory "
    "tool. Greet the student by name if you already know it."
)

async def ask_with_memory(message):
    async with AsyncExitStack() as stack:
        server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"],
             "env": {"MEMORY_FILE_PATH": memory_path}},
            client_session_timeout_seconds=30,
        ))
        memory_agent = Agent(
            name="TAF Agent (with memory)",
            instructions=memory_instructions,
            model=rennes_model,
            mcp_servers=[server],
        )
        try:
            result = await Runner.run(memory_agent, message, max_turns=10)
            return result.final_output
        except Exception as error:
            # The model occasionally loops on its memory tool without ever settling on an
            # answer (see the discussion after Program 5) -- fail gracefully when it does.
            return f"*(the agent got stuck: {error})*"

turn_1 = await ask_with_memory("Hi, I'm Marie, and I'm interested in cybersecurity.")
display(Markdown(turn_1))

turn_2 = await ask_with_memory("What is my name, and what am I interested in?")
display(Markdown(turn_2))

This time `turn_2` really does know Marie's name and interest — not because we resent the conversation, but because the memory MCP server wrote it to `memory/part6_relations.json` after `turn_1`, and read it back before answering `turn_2`. That file *is* the agent's memory: open it yourself (it's plain JSON) and you'll see the stored facts in black and white.

## Putting it together: a TAF advisor with both

Now let's give the same agent both the memory server *and* the `imt_taf_list` tool, so it can recall who it's talking to *and* look up real, current information.

In [ ]:
# Program 5: real information + real memory, together

advisor_instructions = """You are a TAF advisor for IMT Atlantique students.

On EVERY message, in this order:
1. Call the memory tool read_graph (no arguments) first, even if you think you already know
   the answer.
2. If the message contains a new fact about the student (name, interest), store it with the
   memory tool right away.
3. If the student asks about TAF programs, call imt_taf_list to get the current, real list.
4. Answer the student, greeting them by name if read_graph returned one.
"""

async def ask_advisor(message):
    async with AsyncExitStack() as stack:
        server = await stack.enter_async_context(MCPServerStdio(
            {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"],
             "env": {"MEMORY_FILE_PATH": memory_path}},
            client_session_timeout_seconds=30,
        ))
        advisor = Agent(
            name="TAF Advisor",
            instructions=advisor_instructions,
            model=rennes_model,
            mcp_servers=[server],
            tools=[imt_taf_list],
        )
        try:
            result = await Runner.run(advisor, message, max_turns=10)
            return result.final_output
        except Exception as error:
            # With two tool systems to juggle, the model occasionally loops without ever
            # settling on an answer -- see the discussion below.
            return f"*(the agent got stuck: {error})*"

turn_1 = await ask_advisor("Hi, I'm Marie, and I'm interested in cybersecurity.")
display(Markdown(turn_1))

turn_2 = await ask_advisor("What TAF would you recommend for me, and do you remember my name?")
display(Markdown(turn_2))

In our own testing this worked -- `turn_2` remembered Marie and her interest, and correctly recommended the real `TAF Cybersécurité` program -- but it took noticeably more explicit, step-by-step instructions than either tool needed on its own (Programs 2 and 4 each worked fine with a one-line instruction). Juggling two separate tool systems at once (the memory server and our own `imt_taf_list` function) gives the model more to keep track of, and more chances to skip a step -- in one of our test runs, it correctly remembered Marie's name but then invented a plausible-*looking* URL for the recommended TAF instead of using the real one from the tool's output. More tools and more responsibilities don't just add up in capability; they add up in ways an agent can fail, too.

## Key takeaways

* A small local model isn't the bottleneck for using tools *mechanically* -- it's the bottleneck for using them *reliably*. The exact same tool, on a bigger hosted model, works consistently where Ollama did not (Part 5).
* An agent's memory problem (Part 2) is architectural, not a matter of capability: `Runner.run` starts fresh every time, no matter how good the model is, unless something external persists facts between calls.
* MCP packages a tool (including something as generic as "remember facts") as a small, separate server your agent framework can launch and use, instead of writing the same kind of tool by hand every time.
* A memory MCP server backed by a real file (a small database) fixes the amnesia problem properly: facts survive between completely separate `Runner.run` calls, without resending the whole conversation.
* Combining several tools on one agent multiplies what it has to get right, not just what it can do -- expect to need more explicit, step-by-step instructions, and expect it to still occasionally drop one of the steps.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `requests.get`, `Agent`, `Runner.run`, and `@function_tool`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `BeautifulSoup(html, "html.parser")` (`bs4`) | an HTML string, a parser name | a parsed document you can search | Program 1 |
| `soup.find(tag, attrs)` (`bs4`) | a tag name, a dict of attributes to match | the first matching element (or `None`) | Program 1 |
| `element.find_all(tag)` (`bs4`) | a tag name | a list of every matching element inside it | Program 1 |
| `MCPServerStdio(params, client_session_timeout_seconds=)` (`agents.mcp`) | a dict describing how to launch the server (`command`, `args`, `env`), a timeout | an async context manager exposing that server's tools to an `Agent` | Programs 4, 5 |
| `AsyncExitStack()` (`contextlib`) | none | an async context manager that cleanly closes every resource entered into it, in reverse order | Programs 4, 5 |